# 01 — Basic usage

OGC API - Environmental Data Retrieval (EDR) is a query-API for
environmental data (weather, ocean, climate, air quality). The 1.1 spec
defines several query types — `edr-xarray` implements the `/cubes`
query, which returns a regular grid as CoverageJSON.

This notebook walks through the smallest possible workflow:

1. Point xarray at an EDR collection URL.
2. Open the collection as an `xarray.Dataset`.
3. Inspect dimensions, variables, coordinates, attributes.
4. Trigger a single lazy fetch by reading `.values`.
5. Close the dataset when finished.

In [ ]:
# Replace with your EDR collection URL
collection_url = "https://edr.example.com/collections/temperature_2m"

## 2. Open the collection

`edr-xarray` registers itself as the `"edr"` xarray engine when imported.
Pointing `xr.open_dataset` at a `/collections/{id}` URL with
`engine="edr"` is all it takes.

In [ ]:
import xarray as xr

import edr_xarray  # registers engine="edr"

ds = xr.open_dataset(
    collection_url,
    engine="edr",
)
ds

## 3. Inspect structure (no fetch)

At this point the library has issued **at most two requests**: one for
the collection metadata, plus an optional probe of the cube endpoint to
discover grid axes. No actual data values have been transferred yet.

In [ ]:
print("dims:       ", dict(ds.dims))
print("data_vars:  ", list(ds.data_vars))
print("coords:     ", list(ds.coords))
print("attrs:      ", dict(ds.attrs))
print()
print("temperature attrs:")
for k, v in ds["temperature"].attrs.items():
    print(f"  {k}: {v}")

## 4. Lazy load values

Calling `.values` (or `.load()`, `.compute()`) on a `DataArray` triggers
a single GET against the cube endpoint. With no slicing, the full grid
is requested.

In [ ]:
arr = ds["temperature"].values
print("shape:", arr.shape)
print("dtype:", arr.dtype)
print()
print("first time slice (y=3, x=3):")
print(arr[0])

## 5. Cleanup

Always close the dataset when you are done.

In [ ]:
ds.close()